# Lab 17 · Decision models: a weighted decision matrix with a scoring model

**Prepared by Mustafa Assaf** · Self-study: Go deeper · 15–20 minute lab

## Decide on evidence, not a gut call

**So far:** the labs chose between `modular monolith` and `microservices` with rules, a council vote, confidence, and memory. Lab 15 framed the qualities to care about; Lab 16 weighted each architect's *voice*.

**New:** a *decision model* — a repeatable way to turn judgements into a choice. Three approaches do the same job very differently:

1. **A vote** counts opinions only: everyone picks an option, and the most popular wins. It ignores *why* and treats every reason as equal.
2. **A weighted decision matrix** scores each option on a few criteria, multiplies each score by the criterion's *weight*, and sums. Weights are explicit, so a change of weight is a change of model, not a guess.
3. **A scoring model (JEV)** supplies the per-option scores, and plain code does the arithmetic. Scores become *data* that any weight can be applied to, without re-judging.

The difference that matters: a vote and an average hide what you assumed; a weighted matrix states it. The model never chooses for you — it checks your weights against the facts. The first three steps need no model at all; Step 4 is live and needs a TypeSafe key.

**At a glance**

![Lab 17 at a glance: Two options (monolith · microservices), then Score criteria (0–4 per option) or JEV scores (Score primitive), then Weights (the team’s priorities), then Weighted total (Σ score × weight), then Winner (higher total wins). Then back to Weighted total: reweight, never re-judge.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/17.svg)

## The idea in context

Choosing between two options usually ends in an argument because each side argues about a different criterion. A weighted decision matrix forces the disagreement into the open: everyone scores every option on the same criteria, then states how much each criterion matters. The winner is arithmetic, so a dispute about the winner becomes a dispute about a single weight — a much smaller, sharper argument.

A scoring model fits the same frame. TypeSafe JEV is a System One model: it answers typed questions with numbers and probabilities instead of free text, so a call returns a score position in a rubric and a confidence. Scores become data, and changing a weight no longer means re-asking the model.

| Key term | Meaning |
| --- | --- |
| **Decision model** | A repeatable procedure that turns judgements into a choice. |
| **Weighted decision matrix** | Option scores per criterion, each multiplied by the criterion's weight and summed. |
| **Weight** | How much one criterion counts toward the choice; the team's stated priority. |
| **System One model** | A model such as TypeSafe JEV that returns typed judgements and probabilities, not prose. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
if not find_spec('typesafe_sdk'):
    %pip install -q 'typesafe-sdk==0.7.2'

## Why a decision model, not a pattern card

The [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide names *agentic* patterns: how several agents work together. A decision model is not a pattern of agents; it is a *technique you apply to judge options*, like Lab 09's evaluation methodology. Lab 16 used the guide's weighted-scoring form of **Voting-Based Cooperation** to combine votes. This lab keeps the same arithmetic but scores *options on criteria* instead, and lets a model like JEV fill in the per-criterion scores.

## Step 1 · A gut call is a hidden average

The team scores each option from 0 to 4 on four criteria: how fast it ships (Delivery), how cheap it is to run (Cost), how much it needs the lean operations team (Operations), and how well it fits the team's skills (Fit). The first array is the modular monolith — slow to ship, cheap, light on operations, familiar. The second is microservices — the opposite. A plain average treats all four criteria as equally important: a gut call you can now see and challenge.

In [ ]:
criteria = ["Delivery", "Cost", "Operations", "Team fit"]
monolith = [2, 4, 4, 4]       # slow to ship, cheap, light on ops, familiar
microservices = [4, 2, 1, 2]   # ships fast, pricier, heavy on ops, new skills

def average(scores):
    return sum(scores) / len(scores)

for name, scores in [("modular monolith", monolith), ("microservices", microservices)]:
    print(f"{name:<17} average {average(scores):.2f}")

## Step 2 · The weighted decision matrix

The average treated every criterion equally, but this team's real constraint is the single operations engineer. Give Operations weight 3 and the other three criteria weight 1: a weighted score is `score * weight`, summed, then divided by the total weight so it stays on the 0–4 scale. The monolith, which barely touches operations, now wins by a wider margin.

In [ ]:
weights = [1, 1, 3, 1]          # Operations is the binding constraint

def weighted(scores):
    return sum(s * w for s, w in zip(scores, weights)) / sum(weights)

table = [(name, weighted(s)) for name, s in
         [("modular monolith", monolith), ("microservices", microservices)]]
for name, score in table:
    print(f"{name:<17} {score:.2f}")
print("winner:", max(table, key=lambda r: r[1])[0])

## Step 3 · Weights are the model; scores are the facts

Now the team grows to eight engineers and can run many services, so Operations is no longer the constraint: its weight drops to 0 and Delivery, the point of going microservices, rises to 3. The *scores are unchanged* — only the weights moved — and the decision flips. That separation is the point of a matrix: change a weight, never re-judge.

In [ ]:
weights_2 = [3, 1, 0, 1]        # a bigger ops run: Delivery now matters most

def weighted_2(scores):
    return sum(s * w for s, w in zip(scores, weights_2)) / sum(weights_2)

table2 = [(name, weighted_2(s)) for name, s in
          [("modular monolith", monolith), ("microservices", microservices)]]
for name, score in table2:
    print(f"{name:<17} {score:.2f}")
print("winner:", max(table2, key=lambda r: r[1])[0])

## Step 4 (live) · Let JEV score the options once

A scoring model turns each criterion into one typed question and returns a number, not a paragraph. First we build the `state` (the facts) and one `Score` question per option–criterion pair. The `rubric` is an ordered list, lowest to highest; JEV returns a position in it. Building the questions needs no network and no key.

In [ ]:
from os import environ
from typesafe_sdk import Score, TypeSafeClient

state = {
    "team": {"size": 3, "can_independently_deploy": False, "ops_engineers": 1},
    "options": {
        "modular monolith": {"delivery_weeks": 6, "monthly_cost": 1200, "services_to_run": 1},
        "microservices": {"delivery_weeks": 4, "monthly_cost": 2200, "services_to_run": 6},
    },
}
rubric = ["clearly worst", "below average", "average", "good", "clearly best"]
scored = {"modular monolith": monolith, "microservices": microservices}
print("built", len(state["options"]), "options and a", len(rubric), "-level rubric")

## Step 5 (live) · Score once, reweight in code

Now the one call: JEV scores the six questions in parallel and we keep only the numbers. The `weighted_2` matrix runs *unchanged* on JEV's answers, so you can change a weight without re-asking the model. Without a `TYPESAFE_API_KEY` the scripted scores stand in and the arithmetic still runs, so this step is optional.

In [ ]:
def ask():
    questions = {f"{o}:{c}": Score(instructions=f"Rate {o} on {c}. Higher is better.", criteria=rubric)
                 for o in state["options"] for c in criteria}
    with TypeSafeClient() as client:
        return client.system_one(state=state, questions=questions)

if environ.get("TYPESAFE_API_KEY"):
    resp = ask()
    scored = {o: [resp.scores[f"{o}:{c}"].score for c in criteria] for o in state["options"]}
else:
    print("No TYPESAFE_API_KEY; using scripted scores for this cell.")

for name, scores in scored.items():
    print(f"{name:<17} {weighted_2(scores):.2f}")

## What you should see

Step 1's plain average gives the monolith 3.5 against microservices' 2.25, so the monolith leads — but that average hides the operations constraint. Step 2 weights Operations by 3 and the same two score vectors give the monolith 3.67 and microservices 1.83, so the monolith's lead grows. Step 3 moves the weight onto Delivery and off Operations, and with the scores untouched the winner flips: microservices 3.2 beats the monolith 2.8. Step 5 runs that same `weighted_2` over JEV's scored answers (or the scripted stand-in), so you can change one weight and watch the choice move without re-judging.

## Try it

1. Add a fifth criterion, `Security`, scoring the monolith 3 and microservices 4, and give it weight 2. Which option wins now, and does that change when you drop its weight to 0 and raise `Cost` to 3?
2. A `Score` answer carries a `confidence`. Print `resp.scores[...].confidence` and pull any answer below 0.5 down toward the average of the others. Why does a low-confidence judgement deserve its own weight?
3. Two people disagree on weights. Write both weight vectors, run both matrices, and find the single weight that flips the winner. What does that tell you to negotiate first?
4. **Use it in your council.** For one scenario, write three criteria, weight them, and score your two options by hand — then have JEV score them and compare the matrices. Where do they agree, and which differences come from weights rather than scores?

## Takeaway

A weighted decision matrix separates *scores* (the facts, or a model's judgement) from *weights* (the team's priorities). A vote, an average, and a gut call all hide a weight of some kind; the matrix shows it, so a changed priority is a one-line edit, not a re-argument. A System One model like JEV fills in the scores; the arithmetic stays in plain code.

## Read further

- [TypeSafe System One models](https://docs.typesafe.ai/concepts/system-one)
- [The Score primitive](https://docs.typesafe.ai/primitives/score)
- [Keeney and Raiffa, Decisions with Multiple Objectives (1993)](https://doi.org/10.1017/CBO9781139174084)